In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/Telco_Customer_Churn.csv')


df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].median())

df = df.drop('customerID', axis=1)

print(df.shape)
print(df.columns.tolist())

(7043, 20)
['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']


In [2]:
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})
print(df['Churn'].value_counts())

Churn
0    5174
1    1869
Name: count, dtype: int64


In [3]:
X=df.drop('Churn', axis=1)
y=df['Churn']
num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = X.select_dtypes(include=['object']).columns.tolist()

print(f"Numerical: {len(num_cols)} features: {num_cols}")
print(f"Categorical: {len(cat_cols)} features: {cat_cols}")

Numerical: 4 features: ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']
Categorical: 15 features: ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']


C:\Users\lenovo\AppData\Local\Temp\ipykernel_21320\4127950508.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X.select_dtypes(include=['object']).columns.tolist()


In [4]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(drop='first'), cat_cols)
    ])

In [5]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Train: {X_train.shape}")
print(f"Test: {X_test.shape}")
print(f"Train churn %: {y_train.mean()*100:.1f}%")
print(f"Test churn %: {y_test.mean()*100:.1f}%")

Train: (5634, 19)
Test: (1409, 19)
Train churn %: 26.5%
Test churn %: 26.5%


In [6]:
X_train = X_train.copy()
X_test = X_test.copy()

X_train['tenure_group'] = pd.cut(X_train['tenure'], 
                                  bins=[0,12,24,48,72], 
                                  labels=['New','Mid','Mature','Loyal'])

X_test['tenure_group'] = pd.cut(X_test['tenure'], 
                                 bins=[0,12,24,48,72], 
                                 labels=['New','Mid','Mature','Loyal'])

X_train['monthly_per_tenure'] = X_train['MonthlyCharges'] / (X_train['tenure'] + 1)
X_test['monthly_per_tenure'] = X_test['MonthlyCharges'] / (X_test['tenure'] + 1)

service_cols = ['PhoneService','MultipleLines','InternetService',
                'OnlineSecurity','OnlineBackup','DeviceProtection',
                'TechSupport','StreamingTV','StreamingMovies']

X_train['num_services'] = (X_train[service_cols] != 'No').sum(axis=1)
X_test['num_services'] = (X_test[service_cols] != 'No').sum(axis=1)

print(X_train[['tenure_group','monthly_per_tenure','num_services']].head())

     tenure_group  monthly_per_tenure  num_services
3738       Mature            1.366667             5
3151          Mid            4.693750             3
4860          Mid            2.896429             5
3867       Mature            2.722222             6
3810          New           22.275000             2


In [7]:
import joblib
preprocessor.fit(X_train)

# Save karo
X_train.to_csv('../data/processed/X_train.csv', index=False)
X_test.to_csv('../data/processed/X_test.csv', index=False)
y_train.to_csv('../data/processed/y_train.csv', index=False)
y_test.to_csv('../data/processed/y_test.csv', index=False)
joblib.dump(preprocessor, '../models/preprocessor.pkl')

print("Saved!")

Saved!
